# 04 — Hybrid model: RNN for Rating + Bag-of-Words for Recommended

This notebook combines the strongest model family for each target:

- `Rating` is predicted using an RNN (Embedding + GRU)
- `Recommended` is predicted using a Bag-of-Words + Logistic Regression model

The motivation is that previous experiments showed:
- the RNN performed very well on the ordinal `Rating` target
- the Bag-of-Words model performed strongly on the binary `Recommended` target

The final hybrid submission uses:
- rounded integer predictions for `Rating`
- hard binary predictions for `Recommended`

In [2]:
import warnings
warnings.filterwarnings("ignore")

import re
import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MaxAbsScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

import tensorflow as tf
print(tf.__version__)
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

2.18.1


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("Seed set to:", SEED)
print("GPUs:", tf.config.list_physical_devices("GPU"))

Seed set to: 42
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
train = pd.read_csv("../data/rawdata/train.csv")
test = pd.read_csv("../data/rawdata/test.csv")
sample_sub = pd.read_csv("../data/rawdata/sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_sub.shape)


Train shape: (14091, 10)
Test shape: (9395, 8)
Sample submission shape: (9395, 3)


## 1. Shared preprocessing

We keep preprocessing consistent with previous notebooks.

We create:
- missingness indicators
- light text cleaning
- a combined text field

This combined text is used:
- directly by the RNN tokenizer for `Rating`
- by `CountVectorizer` for the BoW `Recommended` model

In [5]:
def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ")
    text = text.lower()
    text = re.sub(r"[^a-z0-9'\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess_reviews(df):
    df = df.copy()
    
    df["title_missing"] = df["Review_Title"].isna().astype(int)
    df["review_missing"] = df["Review"].isna().astype(int)
    
    df["Review_Title"] = df["Review_Title"].fillna("")
    df["Review"] = df["Review"].fillna("")
    
    df["Review_Title_clean"] = df["Review_Title"].apply(clean_text)
    df["Review_clean"] = df["Review"].apply(clean_text)
    
    df["title_prefix"] = np.where(df["title_missing"] == 1, "missingtitle ", "hastitle ")
    df["review_prefix"] = np.where(df["review_missing"] == 1, "missingreview ", "hasreview ")
    
    df["full_text"] = (
        df["title_prefix"] +
        "title " + df["Review_Title_clean"] + " " +
        df["review_prefix"] +
        "review " + df["Review_clean"]
    ).str.strip()
    
    df["title_len_words"] = df["Review_Title_clean"].str.split().str.len()
    df["review_len_words"] = df["Review_clean"].str.split().str.len()
    df["log_Pos_Feedback_Cnt"] = np.log1p(df["Pos_Feedback_Cnt"])
    
    return df

In [6]:
train_p = preprocess_reviews(train)
test_p = preprocess_reviews(test)

In [7]:
train_p[[
    "Review_Title", "Review",
    "title_missing", "review_missing",
    "title_len_words", "review_len_words",
    "full_text"
]].head(10)

,Review_Title,Review,title_missing,review_missing,title_len_words,review_len_words,full_text
0,Cute fall/holiday top,"Love this top! the quality is magnificent and the pattern is even cuter in person. i'm 5'4"", 125 lbs and bought my normal size s which fits excellently. it's pretty flouncy and roomy, which is nic...",0,0,4,63,hastitle title cute fall holiday top hasreview review love this top the quality is magnificent and the pattern is even cuter in person i'm 5'4 125 lbs and bought my normal size s which fits excell...
1,,,1,1,0,0,missingtitle title missingreview review
2,Disappointed,"Sleeves were tight, was difficult to put on ?. for the price, you want to love the shirt. sending it back.",0,0,1,20,hastitle title disappointed hasreview review sleeves were tight was difficult to put on for the price you want to love the shirt sending it back
3,Gorgeous detailing,"I never write reviews but this clothe is so fantastic i felt compelled to write one. it has unbelievably gorgeous detailing, from delicate beading and sequins on the bodice to the three layers of ...",0,0,2,68,hastitle title gorgeous detailing hasreview review i never write reviews but this clothe is so fantastic i felt compelled to write one it has unbelievably gorgeous detailing from delicate beading ...
4,Cute and comfortable tee!,Love this tshirt! casual but can be clotheed up with wedges and a scarf!,0,0,4,14,hastitle title cute and comfortable tee hasreview review love this tshirt casual but can be clotheed up with wedges and a scarf
5,Disappointed,"I was so smitten by this shirt when i saw it online. i love embroidery and purple is my preferred color. i am a size 10 and decided to order an l. the shoulders fit well, but the rest of it was hu...",0,0,1,96,hastitle title disappointed hasreview review i was so smitten by this shirt when i saw it online i love embroidery and purple is my preferred color i am a size 10 and decided to order an l the sho...
6,,"Love these jeans!! i ordered one pair and liked them so much i ordered another right away! i ordered the 26 regular.\nyes, they are too long (i'm 5'4 1/2""# for flats, but look fantastic with heels...",1,0,0,100,missingtitle title hasreview review love these jeans i ordered one pair and liked them so much i ordered another right away i ordered the 26 regular yes they are too long i'm 5'4 1 2 for flats bu...
7,Very large; lace is kinda coarse,"Thought i'd love this top; i'm a sucker for lace anything from retailer. but i found the lace, while pretty, to be quite rough, which to me indicates a lower quality. and i guess i should've know...",0,0,6,62,hastitle title very large lace is kinda coarse hasreview review thought i'd love this top i'm a sucker for lace anything from retailer but i found the lace while pretty to be quite rough which to ...
8,My new preferred jean!,"Paige denim has done it again with this version of the hoxton high rise. no matter how many times i tell myself that i will find a less expensive pair of jeans, i never can find jeans that fit as ...",0,0,4,101,hastitle title my new preferred jean hasreview review paige denim has done it again with this version of the hoxton high rise no matter how many times i tell myself that i will find a less expensi...
9,Pirate sleeves,"The beadwork is gorgeous, but the sleeves are so puffy, it looks as though you're wearing shoulder pads. the fabric of the shirt isn't that fabulous either.",0,0,2,27,hastitle title pirate sleeves hasreview review the beadwork is gorgeous but the sleeves are so puffy it looks as though you're wearing shoulder pads the fabric of the shirt isn't that fabulous either


In [12]:
## Shared train / validation split
#We use the same split for both submodels so the hybrid evaluation is fair.

stratify_label = train_p["Rating"].astype(str) + "_" + train_p["Recommended"].astype(str)

try:
    train_df, val_df = train_test_split(
        train_p,
        test_size=0.2,
        random_state=SEED,
        stratify=stratify_label
    )
except ValueError:
    train_df, val_df = train_test_split(
        train_p,
        test_size=0.2,
        random_state=SEED,
        stratify=train_p["Recommended"]
    )

print("Train split:", train_df.shape)
print("Validation split:", val_df.shape)

Train split: (11272, 20)
Validation split: (2819, 20)


In [10]:
X_train_text = train_df["full_text"].values
X_val_text = val_df["full_text"].values
X_test_text = test_p["full_text"].values

y_train_rating = train_df["Rating"].values.astype("float32")
y_val_rating = val_df["Rating"].values.astype("float32")

y_train_rec = train_df["Recommended"].values.astype("float32")
y_val_rec = val_df["Recommended"].values.astype("float32")

In [11]:
def safe_spearman(y_true, y_pred):
    score = spearmanr(y_true, y_pred).statistic
    if pd.isna(score):
        return 0.0
    return float(score)


def evaluate_predictions(y_true_rating, y_pred_rating, y_true_rec, y_pred_rec):
    rating_score = safe_spearman(y_true_rating, y_pred_rating)
    rec_score = safe_spearman(y_true_rec, y_pred_rec)
    mean_score = (rating_score + rec_score) / 2
    
    return pd.DataFrame({
        "target": ["Rating", "Recommended", "Mean"],
        "spearman": [rating_score, rec_score, mean_score]
    })

## 2. RNN branch for `Rating`

In [13]:
MAX_WORDS = 20000
OOV_TOKEN = "<OOV>"

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_val_seq = tokenizer.texts_to_sequences(X_val_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

In [14]:
train_seq_lengths = np.array([len(seq) for seq in X_train_seq])
MAX_LEN = int(np.percentile(train_seq_lengths, 95))
MAX_LEN = min(MAX_LEN, 150)

print("Chosen MAX_LEN:", MAX_LEN)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

print("Vocabulary size:", vocab_size)
print("Train padded shape:", X_train_pad.shape)
print("Validation padded shape:", X_val_pad.shape)

Chosen MAX_LEN: 109
Vocabulary size: 11145
Train padded shape: (11272, 109)
Validation padded shape: (2819, 109)


In [15]:
rating_model = Sequential()
rating_model.add(Input(shape=(MAX_LEN,)))
rating_model.add(Embedding(input_dim=vocab_size, output_dim=64, mask_zero=True))
rating_model.add(GRU(64))
rating_model.add(Dense(32, activation="relu"))
rating_model.add(Dropout(0.3))
rating_model.add(Dense(1, activation="linear"))

rating_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rating_model.summary()

2026-05-09 07:40:41.659841: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-05-09 07:40:41.660661: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-05-09 07:40:41.661516: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
I0000 00:00:1778305241.662122 3451697 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1778305241.663220 3451697 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 109, 64)        │       713,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 740,353 (2.82 MB)

 Trainable params: 740,353 (2.82 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
early_stopping_rating = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history_rating = rating_model.fit(
    X_train_pad,
    y_train_rating,
    epochs=15,
    validation_data=(X_val_pad, y_val_rating),
    batch_size=64,
    verbose=2,
    callbacks=[early_stopping_rating]
)

Epoch 1/15


2026-05-09 07:41:18.337149: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


177/177 - 8s - 44ms/step - loss: 2.8506 - mae: 1.2873 - val_loss: 1.3083 - val_mae: 0.9455
Epoch 2/15
177/177 - 5s - 30ms/step - loss: 1.5501 - mae: 0.9946 - val_loss: 1.2344 - val_mae: 0.8961
Epoch 3/15
177/177 - 6s - 31ms/step - loss: 1.4936 - mae: 0.9756 - val_loss: 1.2482 - val_mae: 0.9335
Epoch 4/15
177/177 - 5s - 31ms/step - loss: 1.0434 - mae: 0.7990 - val_loss: 0.5941 - val_mae: 0.6163
Epoch 5/15
177/177 - 6s - 35ms/step - loss: 0.7007 - mae: 0.6569 - val_loss: 0.5194 - val_mae: 0.5570
Epoch 6/15
177/177 - 7s - 39ms/step - loss: 0.6467 - mae: 0.6314 - val_loss: 0.5096 - val_mae: 0.5106
Epoch 7/15
177/177 - 7s - 41ms/step - loss: 0.5919 - mae: 0.6015 - val_loss: 0.5031 - val_mae: 0.5173
Epoch 8/15
177/177 - 5s - 31ms/step - loss: 0.5373 - mae: 0.5740 - val_loss: 0.5054 - val_mae: 0.5381
Epoch 9/15
177/177 - 6s - 32ms/step - loss: 0.4883 - mae: 0.5450 - val_loss: 0.5172 - val_mae: 0.5315
Epoch 10/15
177/177 - 7s - 37ms/step - loss: 0.4574 - mae: 0.5291 - val_loss: 0.5369 - val_ma

In [17]:
val_rating_pred_cont = rating_model.predict(X_val_pad, verbose=0).ravel()
val_rating_pred_cont = np.clip(val_rating_pred_cont, 1, 5)

val_rating_pred_round = np.clip(np.rint(val_rating_pred_cont), 1, 5).astype(int)

## 4. Bag-of-Words branch for `Recommended`

In [18]:
text_feature = "full_text"

cat_features = ["Division", "Department", "Product_Category"]

num_features = [
    "Age",
    "Pos_Feedback_Cnt",
    "log_Pos_Feedback_Cnt",
    "title_missing",
    "review_missing",
    "title_len_words",
    "review_len_words"
]

In [19]:
numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MaxAbsScaler())
])

categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

bow_preprocessor = ColumnTransformer([
    ("text", CountVectorizer(
        min_df=5,
        max_features=30000,
        ngram_range=(1, 2),
        lowercase=False
    ), text_feature),
    ("cat", categorical_preprocessor, cat_features),
    ("num", numeric_preprocessor, num_features)
])

In [20]:
recommended_bow_model = Pipeline([
    ("preprocessor", bow_preprocessor),
    ("model", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="liblinear",
        random_state=SEED
    ))
])

In [21]:
recommended_bow_model.fit(train_df, y_train_rec)
print("BoW recommended model trained.")

BoW recommended model trained.


In [22]:
val_rec_pred_prob_bow = recommended_bow_model.predict_proba(val_df)[:, 1]
val_rec_pred_label_bow = recommended_bow_model.predict(val_df).astype(int)

In [23]:
#  5. Hybrid validation evaluation
# - `Rating` from the RNN
# - `Recommended` from the BoW classifier

hybrid_results = []

rating_options = {
    "rating_continuous_rnn": val_rating_pred_cont,
    "rating_rounded_rnn": val_rating_pred_round
}

rec_options = {
    "rec_probability_bow": val_rec_pred_prob_bow,
    "rec_hard_label_bow": val_rec_pred_label_bow
}

for rating_name, rating_pred in rating_options.items():
    for rec_name, rec_pred in rec_options.items():
        score_df = evaluate_predictions(
            y_val_rating, rating_pred,
            y_val_rec, rec_pred
        )
        mean_score = score_df.loc[score_df["target"] == "Mean", "spearman"].iloc[0]

        hybrid_results.append({
            "rating_prediction_type": rating_name,
            "recommended_prediction_type": rec_name,
            "mean_spearman": mean_score,
            "rating_spearman": score_df.loc[score_df["target"] == "Rating", "spearman"].iloc[0],
            "recommended_spearman": score_df.loc[score_df["target"] == "Recommended", "spearman"].iloc[0]
        })

hybrid_results_df = pd.DataFrame(hybrid_results).sort_values("mean_spearman", ascending=False)
hybrid_results_df

,rating_prediction_type,recommended_prediction_type,mean_spearman,rating_spearman,recommended_spearman
3,rating_rounded_rnn,rec_hard_label_bow,0.698621,0.695754,0.701487
1,rating_continuous_rnn,rec_hard_label_bow,0.698419,0.695351,0.701487
2,rating_rounded_rnn,rec_probability_bow,0.638395,0.695754,0.581036
0,rating_continuous_rnn,rec_probability_bow,0.638194,0.695351,0.581036


In [24]:
best_hybrid_row = hybrid_results_df.iloc[0]
best_hybrid_row

rating_prediction_type         rating_rounded_rnn
recommended_prediction_type    rec_hard_label_bow
mean_spearman                            0.698621
rating_spearman                          0.695754
recommended_spearman                     0.701487
Name: 3, dtype: object

In [25]:
kaggle_valid_hybrid = hybrid_results_df[
    hybrid_results_df["rating_prediction_type"] == "rating_rounded_rnn"
].copy().sort_values("mean_spearman", ascending=False)

kaggle_valid_hybrid

,rating_prediction_type,recommended_prediction_type,mean_spearman,rating_spearman,recommended_spearman
3,rating_rounded_rnn,rec_hard_label_bow,0.698621,0.695754,0.701487
2,rating_rounded_rnn,rec_probability_bow,0.638395,0.695754,0.581036


In [26]:
#Final hybrid full dataset fit part

X_full_text = train_p["full_text"].values
X_test_text = test_p["full_text"].values

final_tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token=OOV_TOKEN)
final_tokenizer.fit_on_texts(X_full_text)

X_full_seq = final_tokenizer.texts_to_sequences(X_full_text)
X_test_seq = final_tokenizer.texts_to_sequences(X_test_text)

X_full_pad = pad_sequences(
    X_full_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad_final = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

final_vocab_size = min(MAX_WORDS, len(final_tokenizer.word_index) + 1)

print("Final vocab size:", final_vocab_size)
print("X_full_pad shape:", X_full_pad.shape)
print("X_test_pad_final shape:", X_test_pad_final.shape)

Final vocab size: 12257
X_full_pad shape: (14091, 109)
X_test_pad_final shape: (9395, 109)


In [27]:
final_rating_model = Sequential()
final_rating_model.add(Input(shape=(MAX_LEN,)))
final_rating_model.add(Embedding(input_dim=final_vocab_size, output_dim=64, mask_zero=True))
final_rating_model.add(GRU(64))
final_rating_model.add(Dense(32, activation="relu"))
final_rating_model.add(Dropout(0.3))
final_rating_model.add(Dense(1, activation="linear"))

final_rating_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

final_rating_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 109, 64)        │       784,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 811,521 (3.10 MB)

 Trainable params: 811,521 (3.10 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
final_early_stopping_rating = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

final_history_rating = final_rating_model.fit(
    X_full_pad,
    train_p["Rating"].values.astype("float32"),
    epochs=15,
    validation_split=0.1,
    batch_size=64,
    verbose=2,
    callbacks=[final_early_stopping_rating]
)

Epoch 1/15
199/199 - 9s - 45ms/step - loss: 3.1265 - mae: 1.3489 - val_loss: 1.2915 - val_mae: 0.9065
Epoch 2/15
199/199 - 7s - 33ms/step - loss: 1.5600 - mae: 0.9970 - val_loss: 1.2835 - val_mae: 0.9198
Epoch 3/15
199/199 - 6s - 31ms/step - loss: 1.5259 - mae: 0.9883 - val_loss: 1.2415 - val_mae: 0.8965
Epoch 4/15
199/199 - 6s - 30ms/step - loss: 1.0949 - mae: 0.8214 - val_loss: 0.5851 - val_mae: 0.6248
Epoch 5/15
199/199 - 6s - 31ms/step - loss: 0.7142 - mae: 0.6644 - val_loss: 0.5215 - val_mae: 0.5769
Epoch 6/15
199/199 - 6s - 31ms/step - loss: 0.6282 - mae: 0.6217 - val_loss: 0.4912 - val_mae: 0.5247
Epoch 7/15
199/199 - 6s - 30ms/step - loss: 0.5749 - mae: 0.5943 - val_loss: 0.4863 - val_mae: 0.5187
Epoch 8/15
199/199 - 6s - 30ms/step - loss: 0.5306 - mae: 0.5670 - val_loss: 0.4950 - val_mae: 0.5223
Epoch 9/15
199/199 - 6s - 30ms/step - loss: 0.5147 - mae: 0.5623 - val_loss: 0.4991 - val_mae: 0.5272
Epoch 10/15
199/199 - 6s - 32ms/step - loss: 0.4867 - mae: 0.5467 - val_loss: 0.51

In [29]:
final_recommended_bow_model = Pipeline([
    ("preprocessor", bow_preprocessor),
    ("model", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="liblinear",
        random_state=SEED
    ))
])

final_recommended_bow_model.fit(train_p, train_p["Recommended"])
print("Final BoW recommended model trained.")

Final BoW recommended model trained.


In [30]:
# Rating from RNN
test_rating_pred_cont_hybrid = final_rating_model.predict(X_test_pad_final, verbose=0).ravel()
test_rating_pred_round_hybrid = np.clip(np.rint(test_rating_pred_cont_hybrid), 1, 5).astype(int)

# Recommended from BoW
test_rec_pred_hybrid = final_recommended_bow_model.predict(test_p).astype(int)

In [31]:
submission_hybrid = pd.DataFrame({
    "Id": test_p["Id"].astype(int),
    "Rating": test_rating_pred_round_hybrid.astype(int),
    "Recommended": test_rec_pred_hybrid.astype(int)
})

submission_hybrid.head()

,Id,Rating,Recommended
0,21403,5,1
1,22553,3,0
2,17436,4,1
3,4293,5,1
4,20149,5,1


In [32]:
print(submission_hybrid.shape)
print(submission_hybrid.dtypes)
print(submission_hybrid.isna().sum())

display(submission_hybrid.head())
display(submission_hybrid["Rating"].value_counts().sort_index())
display(submission_hybrid["Recommended"].value_counts().sort_index())

(9395, 3)
Id             int64
Rating         int64
Recommended    int64
dtype: object
Id             0
Rating         0
Recommended    0
dtype: int64


,Id,Rating,Recommended
0,21403,5,1
1,22553,3,0
2,17436,4,1
3,4293,5,1
4,20149,5,1


Rating
1      27
2     733
3    1230
4    2677
5    4728
Name: count, dtype: int64

Recommended
0    1769
1    7626
Name: count, dtype: int64

In [33]:
submission_hybrid.to_csv("submission_04_hybrid_rnnRating_bowRecommended.csv", index=False)
print("Saved: submission_04_hybrid_rnnRating_bowRecommended.csv")

Saved: submission_04_hybrid_rnnRating_bowRecommended.csv
